# Python Developer Associate Practical Exam: ServerWatch Event Monitoring

ServerWatch is an infrastructure monitoring company that processes server event logs for its clients. The engineering team has started building a Python-based log processing pipeline, but the project is incomplete. Some modules were generated by an AI coding assistant and have not been reviewed. Others are stubbed out, waiting for implementation.

The `event_monitor/` directory contains the project:

```
event_monitor/
    config.py        # severity levels and timestamp format (do not edit)
    utils.py         # shared decorator (do not edit)
    loader.py        # Task 1
    log_parser.py    # Task 2
    processor.py     # Task 3
    pipeline.py      # Task 4
    CONTRIBUTING.md
    data/
        server_alpha.csv
        server_beta.csv
        all_events.csv
        alerts.jsonl
```

Your job is to explore the codebase, fix the bugs, and implement the missing functionality. You will be graded on the contents of the `event_monitor/` directory.


## Setup

Run this cell once before starting.


In [ ]:
import sys

if 'event_monitor' not in sys.path:
    sys.path.insert(0, 'event_monitor')

print('Setup complete.')


: 

# Task 1

Before the pipeline can be used, the data loading layer needs to work reliably. `event_monitor/loader.py` was generated by an AI coding assistant and has not been reviewed. Ensure both functions work as described in their docstrings, and make sure no confidential information remains in the source code.


In [ ]:
# You can use this cell as a working space if you choose. Remember that you will be graded on the contents of the event_monitor/ directory.
from pathlib import Path

project_dir = Path("event_monitor")

print("project_dir:", project_dir)
print("Type:", type(project_dir))
print("Folder exists:", project_dir.exists())


: 

In [ ]:
from pathlib import Path

project_dir = Path("event_monitor")

loader_code = '''"""Log file loading utilities."""

import json
from pathlib import Path

import pandas as pd


def load_csv_logs(file_path):
    """Load a CSV log file and return a pandas DataFrame.

    You may assume the file, if it exists, is well-formed: correct
    columns, valid severity values, and no missing fields.

    Parameters
    ----------
    file_path : str or Path
        Path to the CSV log file.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns: timestamp, server, severity, message

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    return pd.read_csv(file_path)


def load_json_alerts(file_path):
    """Load a JSON-lines alerts file and return a list of dicts.

    Each non-blank line of the file contains a single JSON object.
    Blank lines should be skipped.

    Parameters
    ----------
    file_path : str or Path
        Path to the .jsonl alerts file.

    Returns
    -------
    list[dict]
        List of alert records.

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    ValueError
        If a non-blank line contains invalid JSON.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    alerts = []

    with file_path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                alerts.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON on line {line_number}"
                ) from exc

    return alerts
'''

(project_dir / "loader.py").write_text(loader_code, encoding="utf-8")

print("Task 1 implementation saved.")


In [ ]:
from loader import load_csv_logs, load_json_alerts

data_dir = Path("event_monitor") / "data"

df = load_csv_logs(data_dir / "server_alpha.csv")
alerts = load_json_alerts(data_dir / "alerts.jsonl")

print("CSV rows:", len(df))
print("CSV columns:", list(df.columns))
print("Alerts:", len(alerts))


# Task 2

The monitoring team needs to parse event timestamps across different timezone offsets, extract structured error codes from raw log messages, and count events by severity. `event_monitor/log_parser.py` contains three function stubs with docstrings describing the expected behaviour. Implement all three functions.


In [ ]:
# You can use this cell as a working space if you choose. Remember that you will be graded on the contents of the event_monitor/ directory.
from pathlib import Path

project_dir = Path("event_monitor")

log_parser_code = '''"""Log parsing utilities."""

from datetime import datetime, timedelta
from collections import Counter
import re

from config import TIMESTAMP_FORMAT
from utils import log_execution


def parse_timestamps(timestamps, utc_offset_hours=0):
    """Parse timestamp strings and apply a UTC offset.

    The offset is a simple arithmetic adjustment added to the parsed
    (naive) datetimes. This is not timezone-aware conversion: no DST
    or timezone localisation is applied.

    Parameters
    ----------
    timestamps : list[str]
        Timestamp strings in "%Y-%m-%d %H:%M:%S" format.
    utc_offset_hours : int or float
        Hours to offset. Positive shifts forward, negative shifts back.

    Returns
    -------
    list[datetime]
        Parsed and offset datetime objects.

    Example
    -------
    >>> parse_timestamps(["2025-01-15 10:30:00"], utc_offset_hours=2)
    [datetime.datetime(2025, 1, 15, 12, 30)]
    """
    offset = timedelta(hours=utc_offset_hours)

    return [
        datetime.strptime(timestamp, TIMESTAMP_FORMAT) + offset
        for timestamp in timestamps
    ]


def extract_error_codes(messages):
    """Extract error codes from log messages.

    Error codes follow the pattern: 2-4 uppercase letters, a hyphen,
    2-4 uppercase letters, a hyphen, then exactly two digits.

    Each message contains at most one error code. Messages without a
    matching code are skipped.

    Parameters
    ----------
    messages : list[str]
        Raw log message strings.

    Returns
    -------
    list[str]
        Error codes, in the order they appear.

    Example
    -------
    >>> extract_error_codes(["Disk failure [ERR-DISK-01]", "All clear"])
    ['ERR-DISK-01']
    >>> extract_error_codes(["lowercase err-conn-01", "no code"])
    []
    """
    pattern = re.compile(r"\\b[A-Z]{2,4}-[A-Z]{2,4}-\\d{2}\\b")

    error_codes = []

    for message in messages:
        match = pattern.search(message)

        if match:
            error_codes.append(match.group(0))

    return error_codes


@log_execution
def count_by_severity(events, severity_field="severity"):
    """Count events grouped by a severity field.

    This function is decorated with @log_execution from utils.py.
    Do not remove the @log_execution decorator.

    Parameters
    ----------
    events : list[dict]
        Event records containing at least a severity field.
    severity_field : str
        Key name for the severity value in each dict.

    Returns
    -------
    Counter
        Counts keyed by severity level.
    """
    return Counter(event[severity_field] for event in events)
'''

(project_dir / "log_parser.py").write_text(
    log_parser_code,
    encoding="utf-8"
)

print("Task 2 implementation saved successfully.")


In [ ]:
from log_parser import parse_timestamps

timestamps = [
    "2025-01-15 10:30:00",
    "2025-01-16 18:45:30"
]

result = parse_timestamps(timestamps, utc_offset_hours=2)

print(result)


In [ ]:
result = parse_timestamps(
    ["2025-01-15 10:30:00"],
    utc_offset_hours=-3
)

print(result)


In [ ]:
from log_parser import extract_error_codes

messages = [
    "Disk failure [ERR-DISK-01]",
    "All clear",
    "Connection failed [CONN-NET-22]",
    "lowercase err-conn-01",
    "Invalid code ABC-DEF-123",
    "Server issue [WARN-CPU-99]"
]

codes = extract_error_codes(messages)

print(codes)


In [ ]:
from log_parser import count_by_severity

events = [
    {"severity": "ERROR"},
    {"severity": "INFO"},
    {"severity": "ERROR"},
    {"severity": "CRITICAL"},
    {"severity": "WARNING"},
    {"severity": "ERROR"},
]

counts = count_by_severity(events)

print(counts)


In [ ]:
print("Call count:", count_by_severity.call_count)
print("Last execution time:", count_by_severity.last_execution_time)


# Task 3

Currently the pipeline can only process events that are already loaded into memory. The team needs to process logs from multiple server files at once, prioritising the most severe events. Implement `BatchProcessor` in `event_monitor/processor.py` as described in its docstring.


In [ ]:
# You can use this cell as a working space if you choose. Remember that you will be graded on the contents of the event_monitor/ directory.
from pathlib import Path

project_dir = Path("event_monitor")

processor_code = '''"""Event processing classes."""

import pandas as pd
from config import SEVERITY_LEVELS


class EventProcessor:
    """Filter server events by minimum severity level."""

    def __init__(self, events, min_severity="INFO"):
        self.events = events
        self.min_severity = min_severity
        self._processed = None

    def process(self):
        """Filter events at or above min_severity, sorted by timestamp."""
        threshold = SEVERITY_LEVELS.get(self.min_severity, 0)
        self._processed = [
            e for e in self.events
            if SEVERITY_LEVELS.get(e.get("severity", ""), 0) >= threshold
        ]
        self._processed.sort(key=lambda e: e.get("timestamp", ""))
        return self._processed


class AlertProcessor(EventProcessor):
    """Process events and flag alerts."""

    def __init__(self, events, min_severity="WARNING"):
        super().__init__(events, min_severity)

    def process(self):
        filtered = super().process()
        for event in filtered:
            event["is_alert"] = event.get("severity", "") in ("ERROR", "CRITICAL")
        return filtered


class BatchProcessor(EventProcessor):
    """Process events loaded from multiple CSV log files.

    You may assume each CSV file is well-formed: correct columns,
    valid severity values, and no missing fields.

    Parameters
    ----------
    file_paths : list[str]
        Paths to CSV log files.
    min_severity : str
        Minimum severity to include. Default is "INFO".

    The process() method should:
    1. Load each CSV file using pd.read_csv.
    2. Convert each DataFrame to a list of dicts.
    3. Combine all events into self.events.
    4. Filter to events at or above min_severity.
    5. Sort the filtered events by severity descending (most severe
       first), then by timestamp ascending (earliest first) within
       each severity level.
    Return the filtered, sorted list.

    Also implement:
    - __len__: return the number of processed events. Return 0 if
      process() has not yet been called.
    - __repr__: return a string in the format
      "BatchProcessor(<n> files, <m> events processed)"
      where <n> is the number of file paths and <m> is the number of
      processed events (0 if not yet processed).
    """

    def __init__(self, file_paths, min_severity="INFO"):
        super().__init__(events=[], min_severity=min_severity)
        self.file_paths = list(file_paths)

    def process(self):
        """Load, filter, combine, and sort events from all CSV files."""
        all_events = []

        for file_path in self.file_paths:
            data = pd.read_csv(file_path)
            events = data.to_dict(orient="records")
            all_events.extend(events)

        self.events = all_events

        threshold = SEVERITY_LEVELS.get(self.min_severity, 0)

        self._processed = [
            event
            for event in self.events
            if SEVERITY_LEVELS.get(
                event.get("severity", ""), 0
            ) >= threshold
        ]

        self._processed.sort(
            key=lambda event: (
                -SEVERITY_LEVELS.get(
                    event.get("severity", ""), 0
                ),
                event.get("timestamp", "")
            )
        )

        return self._processed

    def __len__(self):
        """Return the number of processed events."""
        if self._processed is None:
            return 0

        return len(self._processed)

    def __repr__(self):
        """Return a string describing the batch processor."""
        return (
            f"BatchProcessor({len(self.file_paths)} files, "
            f"{len(self)} events processed)"
        )
'''

(project_dir / "processor.py").write_text(
    processor_code,
    encoding="utf-8"
)

print("processor.py updated successfully.")


In [ ]:
from processor import BatchProcessor

data_dir = Path("event_monitor") / "data"

file_paths = [
    data_dir / "server_alpha.csv",
    data_dir / "server_beta.csv"
]

batch = BatchProcessor(file_paths)

print(batch)
print("Processed events:", len(batch))


In [ ]:
processed = batch.process()

print("Number of processed events:", len(processed))
print("Batch:", batch)


In [ ]:
print("All events loaded:", len(batch.events))
print("Events after filtering:", len(batch))


In [ ]:
for event in processed:
    print(event["severity"], "|", event["timestamp"], "|", event["server"])


In [ ]:
error_batch = BatchProcessor(
    file_paths,
    min_severity="ERROR"
)

error_events = error_batch.process()

print(error_batch)
print("Severities included:")

for event in error_events:
    print(event["severity"], event["timestamp"])


In [ ]:
allowed = {"ERROR", "CRITICAL"}

assert all(
    event["severity"] in allowed
    for event in error_events
)

print("Minimum-severity filtering passed.")


In [ ]:
test_batch = BatchProcessor(file_paths, min_severity="WARNING")

assert len(test_batch) == 0
assert repr(test_batch) == "BatchProcessor(2 files, 0 events processed)"

test_batch.process()

assert len(test_batch) == len(test_batch._processed)

print(repr(test_batch))
print("BatchProcessor tests passed.")


# Task 4

The team wants a reusable function to summarise a processed event log. In `event_monitor/pipeline.py`, define a dataclass `PipelineResult` with fields `processed_events` (list) and `severity_counts` (`Counter`). Then write a function `run_pipeline` that accepts a `file_path`, loads the CSV using `pd.read_csv`, and returns a `PipelineResult` where `processed_events` is the list of event records (as dicts) and `severity_counts` counts the events by severity.

`run_pipeline` should raise `FileNotFoundError` if the file does not exist and `ValueError` if the file contains no rows.

You can test your function on `event_monitor/data/all_events.csv`.


In [ ]:
# You can use this cell as a working space if you choose. Remember that you will be graded on the contents of the event_monitor/ directory.
from pathlib import Path

project_dir = Path("event_monitor")

pipeline_code = '''"""Pipeline summary utilities."""

from collections import Counter
from dataclasses import dataclass

import pandas as pd


@dataclass
class PipelineResult:
    """Result returned by the event processing pipeline."""

    processed_events: list
    severity_counts: Counter


def run_pipeline(file_path):
    """Load a CSV event log and return a summary.

    Parameters
    ----------
    file_path : str or Path
        Path to the CSV event log.

    Returns
    -------
    PipelineResult
        Processed event records and counts by severity.

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    ValueError
        If the file contains no rows.
    """
    data = pd.read_csv(file_path)

    if data.empty:
        raise ValueError("The file contains no rows.")

    processed_events = data.to_dict(orient="records")
    severity_counts = Counter(data["severity"])

    return PipelineResult(
        processed_events=processed_events,
        severity_counts=severity_counts
    )
'''

(project_dir / "pipeline.py").write_text(
    pipeline_code,
    encoding="utf-8"
)

print("pipeline.py updated successfully.")


In [ ]:
from pipeline import PipelineResult, run_pipeline

data_dir = Path("event_monitor") / "data"

result = run_pipeline(data_dir / "all_events.csv")

print(type(result))
print(result)


In [ ]:
print("Number of events:", len(result.processed_events))
print("Severity counts:", result.severity_counts)


In [ ]:
from dataclasses import is_dataclass
from collections import Counter

assert is_dataclass(PipelineResult)

assert hasattr(result, "processed_events")
assert hasattr(result, "severity_counts")

assert isinstance(result.processed_events, list)
assert isinstance(result.severity_counts, Counter)

print("PipelineResult tests passed.")


In [ ]:
try:
    run_pipeline(data_dir / "file_that_does_not_exist.csv")
except FileNotFoundError:
    print("Missing-file test passed.")


In [ ]:
empty_file = data_dir / "_empty_test.csv"

pd.DataFrame(
    columns=["timestamp", "server", "severity", "message"]
).to_csv(empty_file, index=False)

try:
    run_pipeline(empty_file)
except ValueError:
    print("Empty-file test passed.")
finally:
    empty_file.unlink(missing_ok=True)


In [ ]:
for filename in [
    "loader.py",
    "log_parser.py",
    "processor.py",
    "pipeline.py"
]:
    path = project_dir / filename
    print(filename, "->", path.exists())
